# 基于线性潮流模型的配电网规划 (DNP)

本Jupyter Notebook是MATLAB脚本 `LinearDistFlow_DNP.m` 的Python实现。它使用线性化的DistFlow模型来解决配电网规划问题，该模型用于最优潮流计算。优化模型使用 `cvxpy` 建模，并使用像COPT或SCS这样的求解器求解。

原始模型基于 M. E. Baran 和 F. F. Wu 的工作。

## 1. 导入库和参数

本节导入必要的Python库，并定义电力系统和优化问题的关键参数。

In [ ]:
import os
import numpy as np
import pandas as pd
import cvxpy as cp
import networkx as nx
import matplotlib.pyplot as plt

# Check for COPT solver availability
try:
    import coptpy
    has_copt = True
except ImportError:
    has_copt = False

# --- Parameters ---
N = 16  # number of nodes
L = 33  # number of distribution lines
Sbase = 1e6  # unit: VA
Ubase = 10e3  # unit: V
Ibase = Sbase / (Ubase * np.sqrt(3))  # unit: A
Zbase = Ubase**2 / Sbase # unit: Ω

# Node and line information
# Assuming the excel file is in a subdirectory 'case data'
file_path = os.path.join('case data', '16bus33lines.xlsx')
# Correctly reading the specified range from the Excel sheet.
LineInf = pd.read_excel(file_path, header=None, usecols='F:L', skiprows=2, nrows=33)
NodeInf = pd.read_excel(file_path, header=None, usecols='A:D', skiprows=2, nrows=16)

s = LineInf.iloc[:, 1].values.astype(int)
t = LineInf.iloc[:, 2].values.astype(int)

N_subs = np.array([13, 14, 15, 16])  # Substation nodes
N_loads = np.arange(1, 13)  # Load nodes

v_min = 0.95**2
v_max = 1.05**2
S_max = 12  # max power in any distribution line (MVA)
M = 1e5 # A large number for big-M method

# Costs and Loads
Cost = (LineInf.iloc[:, 6] * LineInf.iloc[:, 3]).values
S_load = NodeInf.iloc[N_loads-1, 3].values
n_pf = 0.8  # power factor
P_load = S_load * n_pf
Q_load = S_load * np.sqrt(1 - n_pf**2)

# Impedance
z = (LineInf.iloc[:, 3] * LineInf.iloc[:, 5] / Zbase).values # line impedance unit:p.u.
n_rx = 1.0
r = z / np.sqrt(1 + n_rx**2)
x = r * n_rx

ValueError: Worksheet named 'F3:L35' not found

## 2. 图定义与可视化

在这里，我们定义配电网络的图结构并创建可视化。

In [ ]:
# --- Graph of the Distribution System ---
G = nx.Graph()
G.add_edges_from(zip(s, t))

# Create incidence matrix
def myincidence(s, t, N):
    num_edges = len(s)
    inc_matrix = np.zeros((N, num_edges))
    for j in range(num_edges):
        inc_matrix[s[j]-1, j] = 1
        inc_matrix[t[j]-1, j] = -1
    return inc_matrix

In = myincidence(s, t, N)

# Plot the Graph
plt.figure(figsize=(12, 8))
pos = {i+1: (NodeInf.iloc[i, 1], NodeInf.iloc[i, 2]) for i in range(N)}

nx.draw(G, pos, with_labels=False, node_color='lightblue', node_size=500)

# Highlight nodes
nx.draw_networkx_nodes(G, pos, nodelist=N_loads, node_color='y', node_size=500)
nx.draw_networkx_nodes(G, pos, nodelist=N_subs, node_color='c', node_shape='s', node_size=700)

# Add labels
nx.draw_networkx_labels(G, pos, font_size=10)

plt.title('16-Bus, 33-Line Distribution System')
plt.show()

## 3. 优化变量

我们使用 `cvxpy` 声明优化问题的决策变量。

In [ ]:
# --- Variable statement ---
v_i = cp.Variable(N)      # v = |U|^2
P_ij = cp.Variable(L)     # Active power flow
Q_ij = cp.Variable(L)     # Reactive power flow
P_shed = cp.Variable(len(N_loads)) # Shedded active load
Q_shed = cp.Variable(len(N_loads)) # Shedded reactive load
g_subs_P = cp.Variable(len(N_subs)) # Active power generation from substations
g_subs_Q = cp.Variable(len(N_subs)) # Reactive power generation from substations
y_ij = cp.Variable(L, boolean=True) # Decision variable for line investment

## 4. 约束条件

本节定义了DNP问题的约束，包括功率平衡、电压限制和线路容量。

In [ ]:
# --- Constraints ---
constraints = []

# 1. Power balance
# Create vectors for power injection at all nodes
P_inj = cp.Variable(N)
Q_inj = cp.Variable(N)

# Load nodes
constraints.append(P_inj[N_loads-1] == -(P_load - P_shed))
constraints.append(Q_inj[N_loads-1] == -(Q_load - Q_shed))

# Substation nodes
constraints.append(P_inj[N_subs-1] == g_subs_P)
constraints.append(Q_inj[N_subs-1] == g_subs_Q)

# Power flow equations
constraints.append(In @ P_ij == P_inj)
constraints.append(In @ Q_ij == Q_inj)

# Shedded load constraints
constraints.append(P_shed >= 0)
constraints.append(Q_shed == P_shed * np.sqrt(1 - n_pf**2) / n_pf)

# 2. Voltage Calculation (Linearized DistFlow)
# v_i - v_j = 2*(r_ij*P_ij + x_ij*Q_ij)
# Using big-M method for conditional constraints based on y_ij
constraints.append(cp.abs(In.T @ v_i - 2 * cp.multiply(r, P_ij) - 2 * cp.multiply(x, Q_ij)) <= (1 - y_ij) * M)

# 3. Voltage limits
constraints.append(v_i[N_subs-1] == v_max) # Fix substation voltage
constraints.append(v_i >= v_min)
constraints.append(v_i <= v_max)

# 4. Line flow limits
# P_ij^2 + Q_ij^2 <= (y_ij * S_max)^2
# This is a set of SOC constraints
# We can also use box constraints as a linear approximation
constraints.append(cp.abs(P_ij) <= y_ij * S_max)
constraints.append(cp.abs(Q_ij) <= y_ij * S_max)

## 5. 目标函数与求解器

目标是最小化总成本，包括新线路的投资成本和切负荷的运营成本。

In [ ]:
# --- Objectives ---
Obj_inv = cp.sum(cp.multiply(Cost, y_ij)) # Investment cost
Obj_ope = M * cp.sum(P_shed) # Operation cost (load shedding penalty)
Obj = Obj_inv + Obj_ope

# --- Solve the problem ---
problem = cp.Problem(cp.Minimize(Obj), constraints)
solver = cp.COPT if has_copt else cp.SCS
problem.solve(solver=solver, verbose=True)

# --- Print Results ---
if problem.status in ["optimal", "optimal_inaccurate"]:
    print('————————— Planning Scheme —————————')
    print(f'Line investment plan: {np.round(y_ij.value, 2)}')
    print('(1/0 means build/do not build the line)')
    print('\\n————————— Planning and Construction Costs —————————')
    print(f'Line construction cost: {Obj_inv.value:.2f} USD')
    print(f'Load shedding cost: {Obj_ope.value:.2f} USD')
    print(f'>> Total planning cost: {problem.value:.2f} USD')

    # Store solution values
    s_y_ij = y_ij.value
    s_v_i = v_i.value
    s_P_ij = P_ij.value
    s_Q_ij = Q_ij.value
    s_P_shed = P_shed.value
else:
    print(f"Problem could not be solved. Status: {problem.status}")

## 6. 结果可视化

最后，我们在网络图上可视化优化结果。这包括与潮流成比例的线宽、节点电压和任何切负荷。

In [ ]:
# --- Plot the results ---
if problem.status in ["optimal", "optimal_inaccurate"]:
    plt.figure(figsize=(15, 10))
    
    # Use the same positions as before
    pos = {i+1: (NodeInf.iloc[i, 1], NodeInf.iloc[i, 2]) for i in range(N)}
    
    # Draw base graph
    nx.draw(G, pos, with_labels=False, node_color='lightgray', node_size=500)
    
    # Draw built lines with widths proportional to power flow
    built_edges = [(s[i], t[i]) for i, y in enumerate(s_y_ij) if y > 0.5]
    edge_widths = [5 * abs(s_P_ij[i]) / max(abs(s_P_ij)) for i, y in enumerate(s_y_ij) if y > 0.5]
    nx.draw_networkx_edges(G, pos, edgelist=built_edges, width=edge_widths, edge_color='blue')

    # Highlight nodes
    nx.draw_networkx_nodes(G, pos, nodelist=N_loads, node_color='y', node_size=500)
    nx.draw_networkx_nodes(G, pos, nodelist=N_subs, node_color='c', node_shape='s', node_size=700)
    
    # Add labels
    nx.draw_networkx_labels(G, pos, font_size=10)

    # Add text for voltages and shed loads
    offset = 0.08
    for i in range(N):
        node = i + 1
        # Voltage
        plt.text(pos[node][0], pos[node][1] + offset, f'{np.sqrt(s_v_i[i]):.3f} pu', 
                 ha='center', fontsize=8, color='green')
        # Shed load
        if node in N_loads:
            shed_idx = np.where(N_loads == node)[0][0]
            if s_P_shed[shed_idx] > 1e-3:
                plt.text(pos[node][0], pos[node][1] - offset, f'Shed: {s_P_shed[shed_idx]:.2f}', 
                         ha='center', fontsize=8, color='red')

    # Add text for power flow
    for i in range(L):
        if s_y_ij[i] > 0.5:
            u, v_node = s[i], t[i]
            mid_x = (pos[u][0] + pos[v_node][0]) / 2
            mid_y = (pos[u][1] + pos[v_node][1]) / 2
            plt.text(mid_x, mid_y, f'{s_P_ij[i]:.2f}+{s_Q_ij[i]:.2f}j', 
                     ha='center', va='center', fontsize=7, color='purple',
                     bbox=dict(facecolor='white', alpha=0.5, edgecolor='none', boxstyle='round,pad=0.1'))

    plt.title('DNP Results: Optimal Network Configuration')
    plt.axis('off')
    plt.show()